In [14]:
# English_Light.py
import os
import json
from pathlib import Path

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# CONFIG
FILE_PATH = r"S:\MCA PRACTICAL\3rd sem\Minor Project\TruthLens\Data\Preprocessed\Preprocessed_english_data.csv"
TEXT_COL = "clean_joined"
LABEL_COL = "isfake"     # original: 1 => fake, 0 => not fake (as you stated)
RANDOM_STATE = 42

TEST_SIZE = 0.30         # 70% train / 30% test
VAL_SIZE = 0.20          # 20% of training used as validation
MIN_TEXT_LEN = 20

OUT_DIR = r"S:\MCA PRACTICAL\3rd sem\Minor Project\TruthLens\models\light\english_light_model"
os.makedirs(OUT_DIR, exist_ok=True)

# LOAD
p = Path(FILE_PATH)
if not p.exists():
    raise FileNotFoundError(f"Data file not found: {FILE_PATH}")

df = pd.read_csv(FILE_PATH, usecols=[TEXT_COL, LABEL_COL], encoding="utf-8", on_bad_lines="skip", low_memory=True)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

# LABELS: raw isfake -> 1 means fake. Internal mapping: 0 = fake, 1 = real
raw_y = df[LABEL_COL].astype(int)
y = (1 - raw_y).astype(int)
X = df[TEXT_COL].astype(str)

# SPLITS
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_STATE)
tr_idx, val_idx = next(sss.split(X_train_all, y_train_all))
X_tr, y_tr = X_train_all.iloc[tr_idx].reset_index(drop=True), y_train_all.iloc[tr_idx].reset_index(drop=True)
X_val, y_val = X_train_all.iloc[val_idx].reset_index(drop=True), y_train_all.iloc[val_idx].reset_index(drop=True)

# BALANCE TRAINING SET (upsample minority)
train_df = pd.DataFrame({TEXT_COL: X_tr.values, "label": y_tr.values})
counts = train_df['label'].value_counts()
if counts.min() != counts.max():
    minority_label = counts.idxmin()
    majority_label = counts.idxmax()
    df_maj = train_df[train_df['label'] == majority_label]
    df_min = train_df[train_df['label'] == minority_label]
    df_min_up = resample(df_min, replace=True, n_samples=len(df_maj), random_state=RANDOM_STATE)
    train_df_bal = pd.concat([df_maj, df_min_up]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    X_tr = train_df_bal[TEXT_COL]
    y_tr = train_df_bal['label']

# VECTORIZER + MODEL
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=5, max_df=0.95, max_features=40000)
base_clf = LinearSVC(random_state=RANDOM_STATE, max_iter=5000, dual=False, class_weight='balanced')
clf = CalibratedClassifierCV(base_clf, cv=3)

X_tr_tfidf = tfidf.fit_transform(X_tr)
clf.fit(X_tr_tfidf, y_tr)

# EVALUATION UTIL
def eval_and_report(X_vec, y_true, model):
    probs = model.predict_proba(X_vec)[:, 1]
    preds = (probs >= 0.5).astype(int)
    r = {}
    r['accuracy'] = float(accuracy_score(y_true, preds))
    r['precision'] = float(precision_score(y_true, preds, zero_division=0))
    r['recall'] = float(recall_score(y_true, preds, zero_division=0))
    r['f1'] = float(f1_score(y_true, preds, zero_division=0))
    try:
        r['roc_auc'] = float(roc_auc_score(y_true, probs))
    except Exception:
        r['roc_auc'] = None
    r['confusion_matrix'] = confusion_matrix(y_true, preds).tolist()
    r['classification_report'] = classification_report(y_true, preds, zero_division=0, output_dict=True)
    return r, probs, preds

X_val_tfidf = tfidf.transform(X_val)
val_metrics, val_probs, val_preds = eval_and_report(X_val_tfidf, y_val, clf)

X_test_tfidf = tfidf.transform(X_test)
test_metrics, test_probs, test_preds = eval_and_report(X_test_tfidf, y_test, clf)

metrics = {'val': val_metrics, 'test': test_metrics}

# SAVE ARTIFACTS
joblib.dump(tfidf, os.path.join(OUT_DIR, 'tfidf_vectorizer.joblib'))
joblib.dump(clf, os.path.join(OUT_DIR, 'classifier.joblib'))

with open(os.path.join(OUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as fh:
    json.dump(metrics, fh, ensure_ascii=False, indent=2)

meta = {
    'text_col': TEXT_COL,
    'label_col': LABEL_COL,
    'raw_label_meaning': 'raw isfake: 1 => fake, 0 => not fake',
    'model_label_mapping': {'fake': 0, 'real': 1},
    'random_state': RANDOM_STATE,
    'n_train': int(len(X_tr)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test))
}
with open(os.path.join(OUT_DIR, 'metadata.json'), 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

# PRINT SUMMARY
print("Validation metrics:")
print(json.dumps(val_metrics, indent=2))
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

print("Sample predictions (first 5 test rows):")
for i in range(min(5, len(X_test))):
    print("TEXT:", X_test.iloc[i][:200].replace("\n"," "))
    print("PROB(real=1):", float(test_probs[i]))
    print("PRED_LABEL (0=fake,1=real):", int(test_preds[i]))
    print("RAW_isfake_original:", int(1 - y_test.iloc[i]))
    print("---")

print("Artifacts saved to:", OUT_DIR)


Validation metrics:
{
  "accuracy": 0.9482484076433121,
  "precision": 0.9432853959894673,
  "recall": 0.9637831125827815,
  "f1": 0.953424096632204,
  "roc_auc": 0.9890967309686267,
  "confusion_matrix": [
    [
      3680,
      280
    ],
    [
      175,
      4657
    ]
  ],
  "classification_report": {
    "0": {
      "precision": 0.9546044098573282,
      "recall": 0.9292929292929293,
      "f1-score": 0.9417786308381318,
      "support": 3960.0
    },
    "1": {
      "precision": 0.9432853959894673,
      "recall": 0.9637831125827815,
      "f1-score": 0.953424096632204,
      "support": 4832.0
    },
    "accuracy": 0.9482484076433121,
    "macro avg": {
      "precision": 0.9489449029233977,
      "recall": 0.9465380209378553,
      "f1-score": 0.9476013637351679,
      "support": 8792.0
    },
    "weighted avg": {
      "precision": 0.9483835869490588,
      "recall": 0.9482484076433121,
      "f1-score": 0.9481788686357839,
      "support": 8792.0
    }
  }
}
Test metric